# 创建collections  
collection本质是一个二维表的结构化集合  
为实现每列一个字段，每行一个实体的管理，需要定义schema。要插入的每个实体都必须符合schema的定义。  
除了schema，还可以定义索引参数提高向量搜索的效率。  
创建一个collection，需要：  
* 创建schema  
* 设置索引（可选，向量field必须）  
* 创建collection

## 创建schema  
schema用来定义collection的数据结构，为每一个field（列）设置名称与数据结构要求。  
- 每个collection必须有一个唯一不重复的主键（常为id，INT64），推荐使用autoid，除非手动设置是有益的  
- enable_dynamic_field 动态字段，允许在插入数据时，插入未在schema中定义的字段。默认建议为False

In [3]:
from pymilvus import MilvusClient, DataType

In [ ]:
client = MilvusClient(uri="http://localhost:19530", token="root:Milvus")

In [ ]:
# 定义schema
schema = MilvusClient.create_schema() # 推荐默认设置（auto_id=False, enable_dynamic_field=False），可在具体field中精确定义
# schema = MilvusClient.create_schema(
#     auto_id=True,
#     enable_dynamic_field=False,
# )

In [ ]:
# 添加主字段（id）
schema.add_field(field_name="my_id", datatype=DataType.INT64, is_primary=True, auto_id=True)
# schema.add_field(field_name="product_id", datatype=DataType.VARCHAR, is_primary=True, auto_id=False, max_length=50)

向量类型包括  
- dense vector：FLOAT_VECTOR、FLOAT16_VECTOR、BFLOAT16_VECTOR、INT8_VECTOR  
- sparse vector: SPARSE_FLOAT_VECTOR  
- binary vector：BINARY_VECTOR(更适合图像处理)  

In [ ]:
# 添加向量
schema.add_field(field_name="dense", datatype=DataType.FLOAT_VECTOR, dim=5) # must dim
schema.add_field(field_name="sparse", datatype=DataType.SPARSE_FLOAT_VECTOR) # no dim

In [ ]:
# 添加标量
schema.add_field(field_name="text", datatype=DataType.VARCHAR, max_length=200) # str
schema.add_field(field_name="myint", datatype=DataType.INT64) # int
schema.add_field(field_name="yes_no", datatype=DataType.BOOL) # bool


In [ ]:
# 添加复合字段
schema.add_field(field_name="json", datatype=DataType.JSON)
schema.add_field(field_name="array", datatype=DataType.ARRAY, element_type=DataType.VARCHAR, max_capacity=5, max_length=512)

In [ ]:
# 删除字段 (在 create_collection 之前)
# schema.fields 是一个列表，直接用列表推导式过滤掉你不想要的字段
schema.fields = [f for f in schema.fields if f.name != "temp_field"]

## 创建索引  
控制field的检索方式与匹配metric  
向量与标量的索引参数不同：  
- 向量必须设置索引：  
    - dense：  
        - metric type：L2（欧氏距离，向量直线距离）, IP（内积）, COSINE（余弦相似度，适合文本）  
        - index type：FLAT, IVF_FLAT, IVF_PQ, IVF_SQ8, HNSW(图), HNSW_SQ, HNSW_PQ, HNSW_PRQ, SCANN ，用于基于 CPU 的 ANN 搜索  
        - 其中：  
            - FLAT：简单的线性扫描，百万级别，100%召回。  
            - IVF_FLAT：IVF 索引 + FLAT 索引。划分nlist个簇，通过比较聚类中心，筛出nprobe个簇，再在nprobe个簇中全部匹配。  
    - binary：  
        - metric type：JACCARD, HAMMING  
        - index type: BIN_FLAT, BIN_IVF_FLAT  
    - sparse:  
        - metric type: IP, BM25  
            - BM25需要搭配参数：params = {"bm25_k1": 1.2, "bm25_b": 0.75}。k1词频饱和度，越小tf上升的边际效应越明显[1.2,2.0]; b文档长度标准化，默认0.75左右，1不归一，0完全归一。
        - index type: SPARSE_INVERTED_INDEX  
            - SPARSE_INVERTED_INDEX需要搭配参数：params = {"inverted_index_algo": "DAAT_MAXSCORE"}。DAAT_MAXSCORE（默认）、DAAT_WAND、TAAT_NAIVE选一
- 标量可选：INVERTED（倒排索引，神）、BITMAP（位图索引）、STL_SORT（INT8、INT16、INT32、INT64、FLOAT、DOUBLE）、Trie（前缀树索引，快速前缀搜索，VARCHAR）  
  
**index type可以直接选择为AUTOINDEX自动判断**

In [ ]:
# 定义索引
index_params = client.prepare_index_params()

In [ ]:
# 标量索引（如主键）
index_params.add_index(
    field_name="my_id",
    index_type="AUTOINDEX"
)

In [ ]:
# 向量索引（如文本dense）
index_params.add_index(
    field_name="dense",
    index_name="dense_index",
    index_type="AUTOINDEX",
    metric_type="COSINE"
)
# index_params.add_index(
#     field_name="dense",
#     index_name="dense_index",
#     index_type="IVF_FLAT",
#     metric_type="COSINE",
#     params={"nlist": 100, "nprobe": 10}
# )
# 文本sparse
# 注意：BM25必须配合方程使用（参考粗召回文档的设置）
index_params.add_index(
    field_name="sparse",
    index_name="sparse_index",
    index_type="SPARSE_INVERTED_INDEX",
    metric_type="BM25",
    params={"inverted_index_algo": "DAAT_MAXSCORE", "bm25_k1": 1.2, "bm25_b": 0.75}
)

In [ ]:
# 可在client中删除索引
# client.drop_index(collection_name="collection1", index_name="dense_index")

## 创建collections  
collections创建包含一系列参数：  
* collection_name: 必填  
* schema：必填  
* index_params：有向量时必填  
* num_shards：可选，切片数量。一般建议2亿数据/100gb便增加一个切片  
* enable_mmap：可选，mmap可以结合内存和磁盘，减少内存占用。内容有限时，建议将需要频繁读取的数据全部载入内存，其它可以启用mmap。mmap有全局、collection、field/索引三种级别，默认都为False。优先级field/索引 > collection > 全局。此处为collection的mmap  
* consistency_level：可选，一致性级别。默认为Bounded。可以理解为可检索数据的时效性。新/返回慢→旧/返回快：Strong > Session > Bounded > Eventual。建议使用Strong保证新数据齐全。一致性有collection、search/query两种级别。优先级为search/query > collection。  
* properties.collection.ttl.seconds：可选，集合的过期时间，单位为秒。默认不开启。开启后每过xx秒会把数据清空

In [15]:
from pymilvus import MilvusClient, DataType

client = MilvusClient(uri="http://localhost:19530", token="root:Milvus")

# 定义schema
schema = MilvusClient.create_schema()
schema.add_field(field_name="my_id", datatype=DataType.INT64, is_primary=True, auto_id=True)
schema.add_field(field_name="dense", datatype=DataType.FLOAT_VECTOR, dim=5)
schema.add_field(field_name="sparse", datatype=DataType.SPARSE_FLOAT_VECTOR)
schema.add_field(field_name="text", datatype=DataType.VARCHAR, max_length=200)

# 定义索引
index_params = client.prepare_index_params()
index_params.add_index(field_name="my_id", index_type="AUTOINDEX")
index_params.add_index(field_name="dense", index_name="dense_index", index_type="AUTOINDEX", metric_type="COSINE")
# 注意：BM25必须配合方程使用，因此此处用IP
index_params.add_index(field_name="sparse", index_name="sparse_index", index_type="SPARSE_INVERTED_INDEX", metric_type="IP", params={"inverted_index_algo": "DAAT_MAXSCORE"})

In [16]:
# 创建collection
client.create_collection(
    collection_name="collection1",
    schema=schema,
    index_params=index_params,
    consistency_level="Strong",
    # shards_num=1,
    # enable_mmap=False,
    # properties={
    #     "collection.ttl.seconds": 999999
    # }
)
# 一般只会设置上面四项

## 加载或释放collections  
将collection加载到内存中，或从内存中释放

In [ ]:
# 加载collection
client.load_collection(collection_name="collection1")
# 释放collection
client.release_collection(collection_name="collection1")
# 查看状态
res = client.get_load_state(collection_name="collection1")
print(res)

## 修改collections
修改名称、部分属性（properties={"collection.ttl.seconds / mmap.enabled / timezone": xxxxxx}）

In [ ]:
# collection重命名
client.rename_collection(old_name="collection1", new_name="collection11")

In [ ]:
# 修改属性
client.alter_collection_properties(
    collection_name="collection1",
    properties={"timezone": "Asia/Shanghai"}
)

### 添加field字段  
可以给collection添加一个新字段：  
* collection_name：必填   
* field_name：必填  
* （和schema定义时一样的参数）  
* nullable：True。新增field时必须设置为True  
* default_value：新增field时，如果设置那么field所有数据为该值

In [ ]:
client.add_collection_field(
    collection_name="collection1",
    field_name="new_int",
    data_type=DataType.INT64,
    nullable=True,
    default_value=50
)

## 删除collections/properties
删除collection或重置某项属性

In [ ]:
# 删除collection
client.drop_collection(collection_name="collection1")

In [ ]:
# 删除重置属性
client.drop_collection_properties(
    collection_name="collection1",
    property_keys=[
        "collection.ttl.seconds"
    ]
)

## 写入数据到collection  
请确保每条数据与collection的schema一致（主键启用autoID后可以不用传入id）  
* 主键启动了autoID时：insert、upsert的数据不传入id（传了也没用），均会自动上传数据并生成主键  
* 主键未启动autoID时：insert、upsert的数据必须包含id。insert遇到相同id会报错，upsert遇到相同id会覆盖原数据。

In [21]:
# 插入数据insert，autoid开启

# 数据
data = [
    {"dense": [0.3580376395471989, -0.6023495712049978, 0.18414012509913835, -0.26286205330961354, 0.9029438446296592], "sparse": {1: 0.5, 100: 0.3, 500: 0.8}, "text": "这是一个例子。"},
    {"dense": [0.19886812562848388, 0.06023560599112088, 0.6976963061752597, 0.2614474506242501, 0.838729485096104], "sparse": {10: 0.1, 200: 0.7, 1000: 0.9}, "text": "这又是一个例子。"}
]
# 插入数据
res = client.insert(
    collection_name="collection1",
    data=data
)
print(res)
# 还可以选择具体的分区
# res = client.insert(
#     collection_name="collection1",
#     data=data,
#     partition_name="partition1"
# )

{'insert_count': 2, 'ids': [463612493562455677, 463612493562455678]}


In [19]:
# 更新数据upsert，autoid未开启
# 数据
data = [
    {"my_id": 0, "dense": [0.3580376395471989, -0.6023495712049978, 0.18414012509913835, -0.26286205330961354, 0.9029438446296592], "sparse": {1: 0.5, 100: 0.3, 500: 0.8}, "text": "这是一个例子。"},
    {"my_id": 1, "dense": [0.19886812562848388, 0.06023560599112088, 0.6976963061752597, 0.2614474506242501, 0.838729485096104], "sparse": {10: 0.1, 200: 0.7, 1000: 0.9}, "text": "这又是一个例子。"}
]
# 更新数据
res = client.upsert(
    collection_name="collection1",
    data=data
)

## 从collection删除数据  
filter条件删除或使用主键删除

In [ ]:
# filter条件删除
res = client.delete(
    collection_name="collection1",
    filter="text like '这又%'", # 模糊查询，开头为“这又”
)
print(res)

{'delete_count': 1}


In [20]:
# 用主键删除
res = client.delete(
    collection_name="collection1",
    ids=[463612493562455672, 463612493562455673]
)
print(res)

{'delete_count': 2}


## 向量匹配search  
输入向量，返回匹配的k个数据，参数：  
- collection_name：检索的collection  
- anns_field：检索的field  
- data：输入的向量。需要是二维列表  
- limit：返回的结果数量  
- search_params.metric_type：选填。搜索metric，优先级比field本身的metric_type高。如search_params = {"metric_type": "COSINE"}  
- output_fields：选填。返回的字段内容。默认只返回id和metric值。如output_fields = ["text", "sparse"]  
- filter：选填。过滤条件。如`filter = "id in [0, 1, 2]"`，或者进行文本匹配，先行缩小向量匹配范围

In [39]:
# 待查询向量
query_vector = [0.3580376395471989, -0.6023495712049978, 0.18414012509913835, -0.26286205330961354, 0.9029438446296592]
# search
res = client.search(
    collection_name="collection1",
    anns_field="dense",
    data=[query_vector],
    limit=3,
    search_params={"metric_type": "COSINE"},
    output_fields=["text"]
)
print(res)

data: [[{'my_id': 463612493562455677, 'distance': 1.0, 'entity': {'text': '这是一个例子。'}}, {'my_id': 463612493562455678, 'distance': 0.6290165185928345, 'entity': {'text': '这又是一个例子。'}}]]


## 条件检索get、query  
* 使用get来根据主键id检索数据  
* 使用query来根据filter条件表达式筛选数据（https://milvus.io/docs/zh/boolean.md）  
可选参数如output_fields、limit

In [26]:
# get
res = client.get(
    collection_name="collection1",
    ids=[463612493562455677, 463612493562455678],
    output_fields=["my_id", "text"]
)
print(res)

data: ["{'my_id': 463612493562455677, 'text': '这是一个例子。'}", "{'my_id': 463612493562455678, 'text': '这又是一个例子。'}"], extra_info: {}


In [ ]:
# query
res = client.query(
    collection_name="collection1",
    filter="text like '%又是%' AND my_id > 463612493562455677", # text包含“又是”，且my_id大于463612493562455677
    limit=1,
    output_fields=["my_id", "text"]
)
print(res)

data: ["{'my_id': 463612493562455678, 'text': '这又是一个例子。'}"], extra_info: {}


## search、query可以搭配filter与schema实现`文本匹配`、`短语匹配`
在schema对字符串对应field定义`enable_analyzer`与`enable_match`后，便可以在search、query用filter实现具体文本匹配与短语匹配  
可以自选是否定义`analyzer_params`：配置分析器。type默认为standard，还可选english、chinese。用于辅助分词和去除停用词，提高匹配效果。  
* filter = "TEXT_MATCH(field_name, text)"  
* filter = "PHARSE_MATCH(field_name, phrase, slop)" # slop为phrase中允许出现的额外编辑数量，最小为0，最大为2。控制phrase灵活度。  
  
启动匹配会同时自动为field构建倒排索引

In [59]:
# 在一个新collection进行该操作
schema = MilvusClient.create_schema()
schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True, auto_id=False)
schema.add_field(field_name="dense", datatype=DataType.FLOAT_VECTOR, dim=5)

# 必须先在schema中定义字段，才能进行文本匹配、短语匹配
# 定义字段时，定义enable_analyzer=True,enable_match=True与analyzer_params（可选）
schema.add_field(
    field_name="text", 
    datatype=DataType.VARCHAR, 
    max_length=1024, 
    enable_analyzer=True, 
    enable_match=True,
    analyzer_params={
        "type": "chinese"
    }
)

client.create_collection(
    collection_name="match_test",
    schema=schema,
    consistency_level="Strong"
)

# index可以随时添加（所有向量field必须有自己的index）
index_params = MilvusClient.prepare_index_params()
index_params.add_index(
    field_name="dense",
    index_name="dense_index",
    index_type="AUTOINDEX",
    metric_type="COSINE"
)
client.create_index(
    collection_name="match_test",
    index_params=index_params
)

# 确保启动加载collection
client.load_collection(collection_name="match_test")

# 先写入两个数据
data = [
    {"id": 0, "dense": [0.3580376395471989, -0.6023495712049978, 0.18414012509913835, -0.26286205330961354, 0.9029438446296592], "text": "这是是一个例子。"},
    {"id": 1, "dense": [0.19886812562848388, 0.06023560599112088, 0.6976963061752597, 0.2614474506242501, 0.838729485096104], "text": "这又是一个例子。"}
]
res = client.upsert(
    collection_name="match_test",
    data=data
)
print(res)

{'upsert_count': 2, 'ids': [0, 1]}


In [60]:
# 以query做文本匹配示例
res = client.query(
    collection_name="match_test",
    filter="not TEXT_MATCH(text, '又') and TEXT_MATCH(text, '例子')",
    output_fields=["text"]
)
print(res)    


data: ["{'text': '这是是一个例子。', 'id': 0}"], extra_info: {}


In [ ]:
# 以query做短语匹配示例（不知道为什么这里只能匹配到一项，可能是分词原因）
res = client.query(
    collection_name="match_test",
    filter="PHRASE_MATCH(text, '这又是', 2)",
    output_fields=["text"]
)
print(res)

data: ["{'text': '这又是一个例子。', 'id': 1}"], extra_info: {}
